# Synthetic Data Pipeline  JSON → CSV → Augmented Training Set

## Purpose
This notebook combines two steps of the data augmentation pipeline:

Converts **LLM-generated synthetic review JSON files** into the standardised CSV format.
```
Input : data/augmented/*.json   (raw LLM output)
Output: data/augmented/*.csv    (one CSV per JSON, same base filename)
```
Merges the original (cleaned) training data with the converted synthetic CSVs to create a larger, more balanced dataset.

```
Output: data/splits/train_augmented.csv` - Combined training set (original + synthetic) 
```

In [ ]:
print("Starting: Loading dependencies and libraries...")

import os
import json
import logging
import re
import yaml
import emoji
import numpy as np
import pandas as pd
from cleantext import clean
from datetime import datetime
from pathlib import Path
from tqdm.auto import tqdm
tqdm.pandas()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
log = logging.getLogger(__name__)

print("Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...


Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.


🕒 Done in 0.63s
✅ Completed: Loading dependencies and libraries.


## Configuration

All paths and aspect names are defined here. These must match the values in `configs/config.yaml`.


In [ ]:
print("Starting: Setting configuration...")

ML_RESEARCH   = os.path.dirname(os.path.abspath(''))
SPLITS_DIR    = Path(ML_RESEARCH) / 'data' / 'splits'     
AUGMENTED_DIR = Path(ML_RESEARCH) / 'data' / 'augmented'  

# Aspect columns — must match model config exactly
ASPECT_COLUMNS = ['stayingpower', 'texture', 'smell', 'price', 'colour', 'shipping', 'packing']
LABELS         = ['positive', 'neutral', 'negative']

print("Completed: Setting configuration.")


⏳ Starting: Setting configuration...
🕒 Done in 0.00s
✅ Completed: Setting configuration.


## Utility Functions

Logger and directory setup helpers used throughout the pipeline.


In [ ]:
print("Starting: Defining utility functions...")

def setup_logger(project_dir: Path, name: str) -> logging.Logger:
    """Set up a logger that writes to both console and a log file."""
    log_dir = Path(project_dir) / 'logs'
    log_dir.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if not logger.handlers:  # Prevent duplicate handlers when running interactively
        fmt = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
        ch  = logging.StreamHandler()
        ch.setFormatter(fmt)
        fh  = logging.FileHandler(log_dir / f'{name}.log', encoding='utf-8')
        fh.setFormatter(fmt)
        logger.addHandler(ch)
        logger.addHandler(fh)
    return logger


def ensure_dirs(project_dir: Path):
    """Create standard output directories if they don't already exist."""
    for sub in ('data/augmented', 'data/splits', 'logs', 'outputs'):
        (Path(project_dir) / sub).mkdir(parents=True, exist_ok=True)

print("Completed: Defining utility functions.")


⏳ Starting: Defining utility functions...
🕒 Done in 0.00s
✅ Completed: Defining utility functions.


## Text Cleaning & Deduplication

Synthetic reviews are cleaned with the **same normalisation** applied to real reviews so both can be mixed without issues.


In [ ]:
print("Starting: Defining text cleaning functions...")

def clean_text(text: str) -> str:
    """
    Lightweight text cleaning for synthetic reviews.
    Steps:
      1. Strip HTML tags   (e.g. <br>, <b>)
      2. Replace emojis with a space (emoji characters cause tokeniser issues)
      3. Collapse whitespace into single spaces
      4. Lowercase + remove URLs, emails, line breaks (via cleantext library)
    """
    text = str(text)
    text = re.sub(r'<.*?>', ' ', text)              # Step 1: strip HTML tags
    text = emoji.replace_emoji(text, replace=' ')   # Step 2: replace emoji with space
    text = re.sub(r'\s+', ' ', text)               # Step 3: collapse whitespace
    text = clean(text, lower=True, no_urls=True, no_emails=True, no_line_breaks=True)  # Step 4
    return text.strip()


def make_signature(row: pd.Series, aspects: list) -> str:
    """
    Build a compact deduplication key by joining all aspect labels.
    Example: 'stayingpower:na|texture:positive|smell:na|colour:negative|...'
    Missing aspect values are represented as 'na'.
    """
    return '|'.join(
        f"{a}:{row[a] if pd.notna(row[a]) and row[a] is not None else 'na'}"
        for a in aspects
    )

print("Completed: Defining text cleaning functions.")


⏳ Starting: Defining text cleaning functions...
🕒 Done in 0.00s
✅ Completed: Defining text cleaning functions.


## Robust JSON Extraction

LLMs may generate files that are not valid JSON at the top level (missing brackets, trailing commas between objects).
The regex approach extracts individual `{...}` objects regardless of the outer structure.


In [ ]:
print("Starting: Defining JSON extraction function...")

def extract_json_objects(file_path, logger):
    """
    Robustly extract all JSON objects { ... } from a file,
    even if the overall structure (commas, brackets) is broken.

    Strategy: regex to find non-nested { ... } blocks.
    Limitation: won't work if review objects contain nested sub-objects.
    This schema is flat, so this is safe here.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    matches = re.findall(r'\{[^{}]+\}', content, re.DOTALL)

    data = []
    for match in matches:
        try:
            obj = json.loads(match)
            data.append(obj)
        except json.JSONDecodeError:
            continue  # Skip objects that can't be parsed

    if not data:
        logger.error(f'No valid JSON objects found in {file_path.name}')
        return None

    logger.info(f'Extracted {len(data)} objects from {file_path.name}')
    return data

print("Completed: Defining JSON extraction function.")


⏳ Starting: Defining JSON extraction function...
🕒 Done in 0.00s
✅ Completed: Defining JSON extraction function.


## Step 1 - Convert JSON Files to CSV

Converts every `.json` file in `data/augmented/` to a matching `.csv`.


In [ ]:
print("Starting: Converting synthetic JSON files to CSV...")

ensure_dirs(Path(ML_RESEARCH))
logger = setup_logger(Path(ML_RESEARCH), 'synthetic_data_pipeline')

json_files = list(AUGMENTED_DIR.glob('*.json'))
if not json_files:
    logger.warning('No JSON files found in data/augmented/ — skipping conversion.')
else:
    for json_path in tqdm(json_files, desc='Converting JSON → CSV'):
        logger.info(f'Processing {json_path.name}...')

        data = extract_json_objects(json_path, logger)
        if data is None:
            continue

        df = pd.DataFrame(data)

        # Normalise column name — some LLM outputs use 'review_text' instead of 'data'
        if 'review_text' in df.columns:
            df = df.rename(columns={'review_text': 'data'})

        # Ensure ALL aspect columns exist (fill missing with None)
        for a in ASPECT_COLUMNS:
            if a not in df.columns:
                df[a] = None

        df = df[df['data'].notna()]  # Drop rows where review text is missing

        df['text_clean'] = df['data'].apply(clean_text)
        print("Running signature transform on all rows...")
        df['signature']  = df.progress_apply(lambda row: make_signature(row, ASPECT_COLUMNS), axis=1)

        cols = ['data'] + ASPECT_COLUMNS + ['text_clean', 'signature']
        df   = df[cols]

        csv_path = json_path.with_suffix('.csv')
        print("Saving CSV...")
        df.to_csv(csv_path, index=False)
        logger.info(f'Saved CSV → {csv_path.name}')

print("Completed: Converting synthetic JSON files to CSV.")


2026-03-20 21:48:45 [WARNING] No JSON files found in data/augmented/ — skipping conversion.
2026-03-20 21:48:45 [WARNING] No JSON files found in data/augmented/ — skipping conversion.


⏳ Starting: Converting synthetic JSON files to CSV...
🕒 Done in 0.00s
✅ Completed: Converting synthetic JSON files to CSV.


## Distribution Helpers

These functions count how many samples exist per class per aspect, and build the Markdown impact report.


In [ ]:
print("Starting: Defining distribution helper functions...")

def get_class_counts(df: pd.DataFrame) -> dict:
    """Return {aspect: {label: count}} for all aspects."""
    dist = {}
    for aspect in tqdm(ASPECT_COLUMNS, desc='Counting aspects'):
        if aspect not in df.columns:
            continue
        vc = df[aspect].value_counts(dropna=True)
        dist[aspect] = {str(lbl): int(vc.get(lbl, 0)) for lbl in LABELS}
    return dist


def build_report(before: dict, after: dict, n_original: int, n_synthetic: int,
                 n_combined: int, synthetic_files: list) -> str:
    """Generate the augmentation_impact.md content as a string."""
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    report  = f"# Data Augmentation Impact Report\n"
    report += f"*Generated on: {now}*\n\n"
    report += "## Summary Statistics\n"
    report += f"- **Original Samples**: {n_original}\n"
    report += f"- **Synthetic Samples added**: {n_synthetic}\n"
    report += f"- **Combined Total**: {n_combined}\n\n"
    report += "## Synthetic Source Files\n"
    for filename, count in synthetic_files:
        report += f"- `{filename}`: {count} samples\n"
    report += "\n"
    report += "## Class Distribution Change\n"
    for aspect in ASPECT_COLUMNS:
        report += f"### {aspect.capitalize()}\n"
        report += "| Label | Before | After | Change |\n"
        report += "|-------|--------|-------|--------|\n"
        for lbl in LABELS:
            b    = before[aspect].get(lbl, 0)
            a    = after[aspect].get(lbl, 0)
            diff = a - b
            report += f"| {lbl} | {b} | {a} | +{diff} |\n"
        report += "\n"
    return report

print("Completed: Defining distribution helper functions.")


⏳ Starting: Defining distribution helper functions...
🕒 Done in 0.00s
✅ Completed: Defining distribution helper functions.


## Step 2 - Build Augmented Training Set

Merges the original training CSV with all synthetic CSVs produced in Step 1.


In [ ]:
print("Starting: Building augmented training set...")

log.info('=' * 65)
log.info('Creating Augmented Training Set')
log.info('=' * 65)

# STEP 1: Load the clean original training data
train_path = SPLITS_DIR / 'train.csv'
if not train_path.exists():
    raise FileNotFoundError('Original training file not found. Run 02_preprocess_and_split.ipynb first!')

print("Reading the dataset from CSV...")
train_df    = pd.read_csv(train_path)
before_dist = get_class_counts(train_df)  # Snapshot class distribution BEFORE augmentation

# STEP 2: Load all synthetic CSVs
csv_files = sorted(AUGMENTED_DIR.glob('*.csv'))
if not csv_files:
    log.warning('No augmented CSV files found — nothing to merge.')
else:
    aug_dfs            = []
    synthetic_file_info = []
    for csv_path in tqdm(csv_files, desc='Merging synthetic data'):
        df = pd.read_csv(csv_path)
        aug_dfs.append(df)
        synthetic_file_info.append((csv_path.name, len(df)))

    # STEP 3: Concatenate, then deduplicate
    synthetic_df = pd.concat(aug_dfs, ignore_index=True)
    synthetic_df = synthetic_df.drop_duplicates(subset=['data'])          # Remove LLM repeats
    synthetic_df = synthetic_df[~synthetic_df['data'].isin(train_df['data'])]  # Prevent leakage

    # STEP 4: Combine — only keep columns present in BOTH DataFrames
    common_cols = [c for c in train_df.columns if c in synthetic_df.columns]
    combined_df = pd.concat([train_df[common_cols], synthetic_df[common_cols]], ignore_index=True)

    # STEP 5: Save the augmented training CSV
    out_path = SPLITS_DIR / 'train_augmented.csv'
    print("Saving augmented training set...")
    combined_df.to_csv(out_path, index=False)
    log.info(f'Saved augmented training set → {out_path}')

    # STEP 6: Generate and save the impact report
    after_dist = get_class_counts(combined_df)
    report     = build_report(before_dist, after_dist, len(train_df),
                              len(synthetic_df), len(combined_df), synthetic_file_info)
    Path(ML_RESEARCH, 'augmentation_impact.md').write_text(report, encoding='utf-8')
    print(f"Writing impact report to {ML_RESEARCH}/augmentation_impact.md...")

print("Completed: Building augmented training set.")
